# Comparing PICLe with Baselines

This notebook reproduces the results of section 5 of the paper. Before running this notebook you need:
1. Pseudo-annotations produced by PICLe
2. Inference results of PICLe for all clusters
3. Aggregated inference results of PICLe
4. Baseline zero-shot inference results
5. Baseline 10-shot inference results with demonstration pool size 10
6. Baseline 10-shot inference results with demonstration pool size 50
7. Baseline 10-shot inference results with demonstration pool size 100

For more information on how to run these experiments, please refer to the README.

In [28]:
import json
import os

import pandas as pd
from utils.config import load_experiment_result_config
from utils.plotting import bar_plot

In [32]:
seed = 12345
results_folder = "outputs"
dataset_list = ["chemprotgene", "chemprotchem", "bc5chem", "bc5disease", "bc2gm"]
picle_demo_retrieval = "specialized_kmeans"
baseline_demo_retrieval = "knn"
baseline_demo_data_size = [10, 50, 100]

In [33]:
setups = []
for dataset in dataset_list:
    setups.append(
        {
            "dataset": dataset,
            "demo_retrieval": picle_demo_retrieval,
            "num_shots": 10,
            "demo_data_size": None,
            "experiment_name": "PICLe",
            "results_path": "",
        }
    )
    setups.append(
        {
            "dataset": dataset,
            "demo_retrieval": "",
            "num_shots": 0,
            "demo_data_size": None,
            "experiment_name": "zero-shot",
            "results_path": "",
        }
    )
    for data_size in baseline_demo_data_size:
        setups.append(
            {
                "dataset": dataset,
                "demo_retrieval": baseline_demo_retrieval,
                "num_shots": data_size,
                "demo_data_size": data_size,
                "experiment_name": f"{baseline_demo_retrieval} from {data_size} gold samples",
                "results_path": "",
            }
        )

sorted_experiments_desc = sorted(os.listdir(results_folder), reverse=True)
for day in sorted_experiments_desc:
    for time in sorted(os.listdir(os.path.join(results_folder, day)), reverse=True):
        hydra_dict = load_experiment_result_config(
            os.path.join(results_folder, day, time, ".hydra"), "hydra"
        )
        runtime_cfg_path = hydra_dict["hydra"]["runtime"]["config_sources"][1][
            "path"
        ].split("/")[-1]

        if runtime_cfg_path in ["baseline", "picle_self_ver"]:
            cfg_dict = load_experiment_result_config(
                os.path.join(results_folder, day, time, ".hydra"), "config"
            )
            dataset = cfg_dict["data"]["dataset"]
            num_shots = cfg_dict.get("demonstration_retrieval", {}).get(
                "num_shots", None
            )
            demo_retrieval = cfg_dict.get("demonstration_retrieval", {}).get(
                "method", None
            )
            demo_data_size = cfg_dict.get("demonstration_retrieval", {}).get(
                "demo_data_size", None
            )

            if dataset in dataset_list:
                for setup in setups:
                    if not setup["results_path"]:
                        if (
                            runtime_cfg_path == "picle_self_ver"
                            and setup["experiment_name"] == "PICLe"
                            and setup["dataset"] == dataset
                            and setup["demo_retrieval"] == picle_demo_retrieval
                        ):
                            setup["results_path"] = os.path.join(
                                results_folder,
                                day,
                                time,
                                "sv_results_ner_iob_evaluation.json",
                            )
                        elif runtime_cfg_path == "baseline":
                            if (
                                setup["experiment_name"]
                                == f"{demo_retrieval} from {setup['demo_data_size']} gold samples"
                                and setup["dataset"] == dataset
                                and setup["num_shots"] == num_shots
                                and setup["demo_retrieval"] == baseline_demo_retrieval
                            ):
                                setup["results_path"] = os.path.join(
                                    results_folder,
                                    day,
                                    time,
                                    "results_ner_iob_evaluation.json",
                                )
                            elif (
                                setup["experiment_name"] == "zero-shot"
                                and setup["dataset"] == dataset
                                and setup["num_shots"] == num_shots
                            ):
                                setup["results_path"] = os.path.join(
                                    results_folder,
                                    day,
                                    time,
                                    "results_ner_iob_evaluation.json",
                                )

for setup in setups:
    if setup["results_path"] == "":
        print(f"Missing results for {setup['experiment_name']}")
        continue

In [34]:
results_dict = {
    "dataset": [],
    "experiment_name": [],
    "Precision": [],
    "Recall": [],
    "Micro F1": [],
}
for setup in setups:
    results = json.load(open(setup["results_path"], "r"))
    results_dict["dataset"].append(setup["dataset"])
    results_dict["experiment_name"].append(setup["experiment_name"])
    results_dict["Precision"].append(results["micro avg"]["precision"])
    results_dict["Recall"].append(results["micro avg"]["recall"])
    results_dict["Micro F1"].append(results["micro avg"]["f1-score"])

results_df = pd.DataFrame(results_dict)

In [ ]:
bar_plot(results_df, "mistral", "Micro F1")